# 01 — Data Inspection & Alignment Verification

Verify that the Phase 1 pipeline produces correct IOB2 alignments:
1. Load raw NER export data and check stats
2. Run fuzzy alignment on sample posts
3. Visualize token ↔ IOB2 tag pairs
4. Check entity distribution across the dataset

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from collections import Counter

from src.data.load_dataset import load_and_validate, print_stats
from src.alignment.fuzzy_matcher import align_post_entities
from src.alignment.token_mapper import (
    get_tokenizer, align_tokens_to_iob2, DEFAULT_LABELS, IGNORE_INDEX
)

## 1. Load Data & Check Stats

Point this at your export file (`data/raw/ner_export.json` or `data/raw/synthetic.json`).

In [ ]:
# Change this path to your data file
DATA_PATH = Path("../data/raw/ner_export.json")

posts, stats = load_and_validate(DATA_PATH)
print_stats(stats, source=DATA_PATH.name)

## 2. Inspect Fuzzy Alignment on Sample Posts

Run the alignment pipeline on a few posts and inspect the character-level matches.

In [ ]:
NUM_SAMPLES = 5

for post in posts[:NUM_SAMPLES]:
    print(f"\n{'='*60}")
    print(f"Post ID: {post.id}")
    print(f"Text: {post.raw_text[:120]}{'...' if len(post.raw_text) > 120 else ''}")
    print(f"Entities to align: {len(post.entities)}")

    aligned, unmatched = align_post_entities(post.raw_text, post.entities)

    for a in aligned:
        print(f"  ✓ [{a.label}] '{a.entity_text}' → '{a.matched_text}' "
              f"[{a.char_start}:{a.char_end}] score={a.score:.0f}")

    for u in unmatched:
        print(f"  ✗ [{u['label']}] '{u['text']}' — no match found")

## 3. Token ↔ IOB2 Tag Visualization

Load the tokenizer and show how subword tokens map to IOB2 labels for a sample post.

In [ ]:
tokenizer = get_tokenizer()

# Pick a sample post
sample = posts[0]
aligned, _ = align_post_entities(sample.raw_text, sample.entities)
result = align_tokens_to_iob2(tokenizer, sample.raw_text, aligned)

print(f"Post: {sample.raw_text}\n")
print(f"{'Token':<20} {'Label':<25} {'Offsets'}")
print("-" * 60)
for token, label_id, offsets in zip(result.tokens, result.labels, result.offset_mapping):
    if label_id == IGNORE_INDEX:
        label_str = "[SPECIAL]"
    else:
        label_str = DEFAULT_LABELS[label_id]

    # Highlight entity tokens
    marker = " ◄" if label_str not in ("O", "[SPECIAL]") else ""
    print(f"{token:<20} {label_str:<25} {offsets}{marker}")

## 4. Entity Distribution Chart

Visualize label distribution across the full dataset.

In [ ]:
import matplotlib.pyplot as plt

labels_sorted = sorted(stats.label_counts.keys())
counts = [stats.label_counts[l] for l in labels_sorted]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels_sorted, counts, color="#4f46e5")
ax.set_xlabel("Count")
ax.set_title("Entity Distribution")
ax.bar_label(bars, padding=3)
plt.tight_layout()
plt.show()

## 5. Batch Alignment Stats

Run alignment on all posts and report success/failure rates.

In [ ]:
total_entities = 0
matched_entities = 0
unmatched_labels = Counter()

for post in posts:
    aligned, unmatched = align_post_entities(post.raw_text, post.entities)
    total_entities += len(post.entities)
    matched_entities += len(aligned)
    for u in unmatched:
        unmatched_labels[u["label"]] += 1

match_rate = matched_entities / total_entities * 100 if total_entities else 0

print(f"Total entities:    {total_entities}")
print(f"Matched:           {matched_entities} ({match_rate:.1f}%)")
print(f"Unmatched:         {total_entities - matched_entities}")
if unmatched_labels:
    print(f"\nUnmatched by label:")
    for label, count in unmatched_labels.most_common():
        print(f"  {label}: {count}")